# Chapter 7: Agent Observability with Strands

This notebook walks through three observability setups for Strands agents:

1. **Local tracing with Jaeger** - see traces on your machine during development
2. **Langfuse integration** - open-source LLM observability platform
3. **Console export** - quick debugging without any external services

All of these use OpenTelemetry (OTel) under the hood. Strands emits OTel spans for every model call, tool invocation, and agent cycle automatically.

## Install Dependencies

You need the `otel` extra to enable OpenTelemetry tracing in Strands.

In [ ]:
%pip install 'strands-agents[otel]' strands-agents-tools

---
## 1. Console Tracing (No External Services)

The simplest way to see what your agent is doing. Traces print directly to stdout.
Good for quick debugging when you do not want to set up a collector.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.telemetry import StrandsTelemetry

# Print all spans to console
strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_console_exporter()

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(model=model, system_prompt="You are a helpful assistant.")

response = agent("What is OpenTelemetry and why does it matter for AI agents?")
print(response)

You should see structured span output above showing:
- An **Agent span** (top level) with total token counts
- **Cycle spans** for each reasoning loop iteration
- **Model invoke spans** with input/output tokens, latency, and model ID
- **Tool spans** if the agent called any tools (with tool name, parameters, and results)

---
## 2. Local Tracing with Jaeger

Jaeger gives you a visual trace UI on your local machine. Run this in a terminal first:

```bash
docker run -d --name jaeger \
  -e COLLECTOR_OTLP_ENABLED=true \
  -p 16686:16686 \
  -p 4317:4317 \
  -p 4318:4318 \
  jaegertracing/all-in-one:latest
```

Then open http://localhost:16686 in your browser to view traces.

In [ ]:
import os
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.telemetry import StrandsTelemetry

# Point at the local Jaeger OTLP endpoint
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:4318"

strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_otlp_exporter()

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(model=model, system_prompt="You are a helpful assistant.")

response = agent("Explain the difference between traces, spans, and metrics in OpenTelemetry.")
print(response)

Go to http://localhost:16686, select the `strands-agents` service, and click **Find Traces**.

You will see a waterfall view showing the full agent execution:
- How long each model call took
- Token counts (input and output) per model invocation
- Tool calls with their parameters and results
- The total duration of the agent invocation

---
## 3. Langfuse Integration

[Langfuse](https://langfuse.com) is an open-source LLM observability platform. It provides cost tracking,
prompt versioning, scoring, and nested trace visualization.

You can use the managed cloud version or self-host it. To get started with the cloud version:

1. Go to [https://cloud.langfuse.com](https://cloud.langfuse.com) (US region) or [https://eu.cloud.langfuse.com](https://eu.cloud.langfuse.com) (EU region) and create a free account.
2. Once logged in, create a new **Project** (e.g., `my-llm-project`).
3. Inside the project, go to **Settings** (gear icon in the left sidebar) and then **API Keys**.
4. Click **Create new API keys**. This gives you a **Public Key** (starts with `pk-lf-`) and a **Secret Key** (starts with `sk-lf-`). Copy both.
5. Set the **Base URL** to `https://us.cloud.langfuse.com` for US or `https://cloud.langfuse.com` for EU, depending on which region you signed up in.

Langfuse has a native integration with Strands via its SDK. Once you set the keys as environment variables below, the Langfuse client automatically instruments your Strands agent through OpenTelemetry.

In [ ]:
%pip install langfuse

In [ ]:
import os

# Get these from your Langfuse project settings
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"  # or your self-hosted URL

In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify the connection works
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready.")
else:
    print("Authentication failed. Check your credentials.")

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(
    model=model,
    system_prompt="You are a document analyst. Summarize documents clearly and concisely."
)

response = agent("What are the key risks in a standard SaaS subscription agreement?")
print(response)

# Make sure traces are flushed to Langfuse
langfuse.flush()

Open your Langfuse dashboard. You should see the trace with:
- The system prompt and user message
- Each model call with token counts and latency
- Any tool invocations with parameters and results
- Estimated cost based on the model pricing

---
## 4. LangSmith Integration

[LangSmith](https://smith.langchain.com) is LangChain's observability and evaluation platform. It was built for the LangChain and LangGraph ecosystem, but it accepts OTel traces from any framework, including Strands.

To get started:

1. Go to [https://smith.langchain.com](https://smith.langchain.com) and create a free account.
2. Once logged in, create a new **Project** (or use the default project).
3. Click your profile icon in the bottom left, then **Settings**, then **API Keys**.
4. Create a new API key. It will start with `lsv2_pt_`. Copy it.

LangSmith accepts OTel traces at `https://api.smith.langchain.com/otel`. You pass your API key and project name in the OTel headers.

In [ ]:
import os

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.smith.langchain.com/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = "x-api-key=lsv2_pt_...,LANGSMITH_PROJECT=my-project"

from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands_tools import http_request, calculator

strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_otlp_exporter()

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(
    model=model,
    system_prompt="You are a helpful research assistant.",
    tools=[http_request, calculator]
)

response = agent("What is 15% of 2,847? Then fetch the homepage of https://httpbin.org/get and tell me what headers it returns.")
print(response)

Open your LangSmith dashboard and navigate to your project. You should see the trace with the same nested structure as Langfuse and Jaeger: agent invocation, model calls, tool executions, token counts, and timing.

LangSmith also provides:
- A **Playground** where you can open any trace, modify the prompt, and re-run it without redeploying
- **Datasets** built from production traces that you can use as regression test suites
- **Annotation queues** for human review and scoring of agent outputs

---
## 5. AgentCore Observability (CloudWatch)

If your agent runs on AgentCore Runtime, observability is built in. But you can also send traces from agents running anywhere (your laptop, Lambda, EC2) to CloudWatch using the AWS Distro for OpenTelemetry (ADOT).

This section shows how to set up a Strands agent running locally to send traces to CloudWatch GenAI Observability.

### Prerequisites

1. AWS credentials configured (`aws configure`)
2. Enable **Transaction Search** in CloudWatch: https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html#CloudWatch-Transaction-Search-EnableConsole
3. A CloudWatch log group and log stream (we will create these below)

In [ ]:
%pip install aws-opentelemetry-distro python-dotenv

### Create a CloudWatch Log Group and Log Stream

AgentCore Observability needs a log group and stream to receive the OTel data.

In [ ]:
import boto3

LOG_GROUP = "agents/observability-demo"
LOG_STREAM = "default"

logs_client = boto3.client("logs")

try:
    logs_client.create_log_group(logGroupName=LOG_GROUP)
    print(f"Created log group: {LOG_GROUP}")
except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"Log group already exists: {LOG_GROUP}")

try:
    logs_client.create_log_stream(logGroupName=LOG_GROUP, logStreamName=LOG_STREAM)
    print(f"Created log stream: {LOG_STREAM}")
except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"Log stream already exists: {LOG_STREAM}")

### Write the Agent to a Python File

ADOT instruments your agent at the process level using the `opentelemetry-instrument` command.
This means the agent needs to run as a standalone Python script, not inside a notebook cell.
We write it to a file and then run it with the instrumentation wrapper.

In [ ]:
%%writefile agentcore_agent.py
import os
from strands import Agent, tool
from strands.models import BedrockModel


@tool
def lookup_account(account_id: str) -> str:
    """Look up account details by account ID."""
    # Simulated account data
    accounts = {
        "ACC-001": {"name": "Acme Corp", "tier": "Enterprise", "balance": 142500.00},
        "ACC-002": {"name": "Startup Inc", "tier": "Growth", "balance": 28750.00},
    }
    account = accounts.get(account_id)
    if account:
        return f"Account {account_id}: {account['name']}, Tier: {account['tier']}, Balance: ${account['balance']:,.2f}"
    return f"Account {account_id} not found."


@tool
def calculate_discount(amount: float, tier: str) -> str:
    """Calculate discount based on customer tier."""
    rates = {"Enterprise": 0.15, "Growth": 0.10, "Starter": 0.05}
    rate = rates.get(tier, 0.0)
    discount = amount * rate
    return f"{tier} tier discount: {rate*100:.0f}% off ${amount:,.2f} = ${discount:,.2f} savings"


model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
)

agent = Agent(
    model=model,
    system_prompt="You are a customer account assistant. Use the available tools to help with account inquiries.",
    tools=[lookup_account, calculate_discount],
)

response = agent("Look up account ACC-001 and calculate what discount they would get on a $50,000 purchase.")
print(response)

### Configure ADOT Environment Variables

These environment variables tell the ADOT SDK where to send traces and how to identify your agent in CloudWatch.

In [ ]:
%%writefile .env.agentcore
OTEL_PYTHON_DISTRO=aws_distro
OTEL_PYTHON_CONFIGURATOR=aws_configurator
OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf
OTEL_TRACES_EXPORTER=otlp
OTEL_EXPORTER_OTLP_LOGS_HEADERS=x-aws-log-group=agents/observability-demo,x-aws-log-stream=default,x-aws-metric-namespace=bedrock-agentcore
OTEL_RESOURCE_ATTRIBUTES=service.name=ch7-account-agent
AGENT_OBSERVABILITY_ENABLED=true

### Run the Agent with ADOT Instrumentation

The `opentelemetry-instrument` command wraps your Python script and automatically captures traces for every model call, tool invocation, and HTTP request the agent makes. The ADOT SDK reads the environment variables above and sends the trace data to CloudWatch.

In [ ]:
import subprocess
import os
from dotenv import dotenv_values

# Load the ADOT env vars
env = {**os.environ, **dotenv_values(".env.agentcore")}

result = subprocess.run(
    ["opentelemetry-instrument", "python", "agentcore_agent.py"],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

### View Traces in CloudWatch

After running the agent, go to the [CloudWatch GenAI Observability dashboard](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability). You should see your agent listed under the Bedrock AgentCore section.

Drill into the trace and you will see the full span hierarchy: the agent invocation at the top, the model calls with token counts, and the tool calls (`lookup_account` and `calculate_discount`) with their input parameters and results.

This is the same trace data that Jaeger, Langfuse, and LangSmith display, but viewed through CloudWatch's GenAI Observability interface.

### Clean Up

Delete the log group when you are done to avoid ongoing CloudWatch charges.

In [ ]:
# Uncomment to delete the log group
# logs_client.delete_log_group(logGroupName=LOG_GROUP)
# print(f"Deleted log group: {LOG_GROUP}")